# Caracterización Integral y Cálculo de Área TOTAL del Regulador LDO (Sky130)
---
Este cuaderno realiza la caracterización completa del Regulador de Baja Caída de Tensión (**LDO**) definido en `ldo_tb.sch` y `LDO.sch`, leyendo las simulaciones y netlists de `/headless/.xschem/simulations/`:

1. **Lector Nativo en Python Estándar de Archivos RAW:**
   - Carga y procesamiento directo de `LDO_ac.raw`, `LDO_trans.raw` y `LDO_op.raw`.

2. **Ganancia de Lazo Abierto $T(s)$ y Estabilidad (Respuesta AC):**
   - Extracción de **Ganancia de Lazo DC ($A_{v0}$)**, **Frecuencia de Corte ($f_{-3dB}$)**, **Frecuencia de Ganancia Unitaria ($GBW$)** y **Margen de Fase ($PM$)**.
   - Diagrama de Bode interactivo en Plotly con **escala logarítmica explícita en ambos subgráficos** (magnitud de lazo y fase de lazo).

3. **Respuesta Transitoria a Escalón de Carga (.TRAN):**
   - Análisis de la tensión de salida $V_{out}(t)$ ante escalones de corriente de carga $I_{load}(t)$ (de $1\text{ mA}$ a $50\text{ mA}$), calculando caídas transitorias (*undershoot/overshoot*) y regulación.

4. **Punto de Operación (.OP) del Transistor de Paso y Amplificador:**
   - Verificación de la región de operación del transistor de paso PMOS LVT ($M_1$) y de todos los transistores del amplificador de error.

5. **Cálculo del ÁREA TOTAL de Todo el LDO:**
   - Escaneo completo de `/headless/.xschem/simulations/LDO.spice` y `OTA.spice` para sumar el **área física de TODOS los bloques del regulador**: Transistor de paso $M_1$, resistencias divisoras de realimentación $R_1, R_2$, capacitor de compensación del LDO $C_1$, y todos los transistores, resistencias y capacitores del amplificador de error OTA.



In [1]:
# ==============================================================================
# ENCABEZADO: Respuesta en Frecuencia del Lazo T(s) y Diagrama de Bode (Plotly)
# DESCRIPCIÓN: Lee LDO_ac.raw desde /headless/.xschem/simulations/ usando Python
#              estándar y aplica eje logarítmico a ambos subgráficos de lazo.
# ==============================================================================

import subprocess
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── 1. Rutas de Simulaciones ──────────────────────────────────────────────────
sim_dir = "/headless/.xschem/simulations"
xschem_dir = "/foss/designs/LDO-sky130/xschem"
sch_file = f"{xschem_dir}/ldo_tb.sch"

if not os.path.exists(sim_dir):
    os.makedirs(sim_dir, exist_ok=True)

subprocess.run(f"export PDK=sky130A && /foss/tools/bin/xschem -n -q -o {sim_dir} {sch_file}", shell=True)
subprocess.run(f"cd {sim_dir} && /foss/tools/bin/ngspice -b {sim_dir}/ldo_tb.spice", shell=True)

# ── 2. Lector Nativo RAW ASCII de NGspice en Python Estándar ───────────────────
def read_ngspice_raw_ascii(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()
        
    num_vars = 0
    is_complex = False
    var_names = []
    idx = 0
    
    while idx < len(lines):
        line = lines[idx].strip()
        if line.startswith('Flags:'):
            if 'complex' in line.lower():
                is_complex = True
        elif line.startswith('No. Variables:'):
            num_vars = int(line.split(':')[1].strip())
        elif line.startswith('Variables:'):
            idx += 1
            for _ in range(num_vars):
                parts = lines[idx].strip().split()
                var_names.append(parts[1])
                idx += 1
            continue
        elif line.startswith('Values:'):
            idx += 1
            break
        idx += 1
        
    data = {v: [] for v in var_names}
    current_pt = 0
    
    while idx < len(lines):
        line = lines[idx].strip()
        if not line:
            idx += 1
            continue
        parts = line.split('\t')
        val_str = parts[1].strip() if len(parts) >= 2 else line
        
        if is_complex:
            if ',' in val_str:
                r, i = map(float, val_str.split(','))
                val = complex(r, i)
            else:
                val = complex(float(val_str), 0.0)
        else:
            val = float(val_str)
            
        var_name = var_names[current_pt % num_vars]
        data[var_name].append(val)
        current_pt += 1
        idx += 1
        
    for k in data:
        data[k] = np.array(data[k])
    return data

# ── 3. Procesamiento de Ganancia de Lazo T(s) ──────────────────────────────────
raw_ac_path = os.path.join(sim_dir, "LDO_ac.raw")
if not os.path.exists(raw_ac_path):
    raw_ac_path = os.path.join(xschem_dir, "LDO_ac.raw")

ac_data = read_ngspice_raw_ascii(raw_ac_path)
freq = np.real(ac_data['frequency'])

# Ganancia de lazo T(s) = v(x1.vre) / v(x1.net3)
vre = ac_data['v(x1.vre)']
net3 = ac_data['v(x1.net3)']
t_loop = vre / net3

mag_db = 20 * np.log10(np.abs(t_loop) + 1e-18)
phase_rad = np.angle(t_loop)
phase_unwrap_rad = np.unwrap(phase_rad)
phase_deg = np.rad2deg(phase_unwrap_rad)

# ── 4. Métricas de Estabilidad AC por Interpolación Lineal ────────────────────
av0_db = mag_db[0]
av0_lin = 10**(av0_db / 20.0)

idx_mono = np.where(freq >= 100)[0]
target_3db = av0_db - 3.0

f_3db = float(np.interp(target_3db, mag_db[idx_mono][::-1], freq[idx_mono][::-1]))
f_gbw = float(np.interp(0.0, mag_db[idx_mono][::-1], freq[idx_mono][::-1]))
ph_at_gbw = float(np.interp(f_gbw, freq, phase_deg))

# Margen de fase del LDO
pm = ph_at_gbw if ph_at_gbw > 0 else (180.0 + ph_at_gbw)

print("=== RESULTADOS DE ESTABILIDAD EN LAZO ABIERTO DEL LDO ===")
print(f"Ganancia de Lazo DC (Av0):      {av0_db:.2f} dB ({av0_lin:.2f} V/V)")
print(f"Frecuencia de Corte (-3dB):    {f_3db/1e3:.2f} kHz")
print(f"Ancho de Banda GBW (0 dB):     {f_gbw/1e3:.2f} kHz ({f_gbw/1e6:.2f} MHz)")
print(f"Fase en GBW:                   {ph_at_gbw:.2f}°")
print(f"Margen de Fase (PM):           {pm:.2f}°")

# ── 5. Diagrama de Bode de la Ganancia de Lazo T(s) con Eje Logarítmico en Ambos Subgráficos ──
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    vertical_spacing=0.1,
                    subplot_titles=("Ganancia de Lazo [dB]", "Fase de Lazo [grados]"))

fig.add_trace(go.Scatter(x=freq, y=mag_db, mode='lines', name='Magnitud T(s) [dB]', line=dict(color='#1f77b4', width=2.5)), row=1, col=1)
if not np.isnan(f_gbw):
    fig.add_trace(go.Scatter(x=[f_gbw], y=[0], mode='markers+text', name='GBW',
                             text=[f'GBW: {f_gbw/1e3:.1f} kHz'], textposition='top right',
                             marker=dict(color='green', size=10, symbol='star')), row=1, col=1)

fig.add_trace(go.Scatter(x=freq, y=phase_deg, mode='lines', name='Fase T(s) [°]', line=dict(color='#d62728', width=2.5)), row=2, col=1)

# Aplicar escala logarítmica explícitamente a ambos ejes X
fig.update_xaxes(type="log", row=1, col=1)
fig.update_xaxes(type="log", title_text="Frecuencia [Hz]", row=2, col=1)

fig.update_yaxes(title_text="Magnitud [dB]", row=1, col=1)
fig.update_yaxes(title_text="Fase [°]", row=2, col=1)

fig.update_layout(height=650, title_text="<b>Diagrama de Bode de Ganancia de Lazo del LDO (Sky130)</b>", template="plotly_white")
fig.show()



Using run time directory XSCHEM_SHAREDIR = /foss/tools/xschem/share/xschem
Sourcing /foss/tools/xschem/share/xschem/xschemrc init file
Sourcing /headless/.xschem/xschemrc init file
open_pdks installation: using /foss/pdks
SKYWATER_MODELS: /foss/pdks/sky130A/libs.tech/combined
SKYWATER_STDCELLS: /foss/pdks/sky130A/libs.ref/sky130_fd_sc_hd/spice


setup_tcp_bespice: problems finding a free TCP port
couldn't open socket: address already in use




Note: Compatibility modes selected: hs a


Circuit: ** sch_path: /foss/designs/ldo-sky130/xschem/ldo_tb.sch

option SCALE: Scale is set to 1e-06 for instance and model parameters
Doing analysis at TEMP = 27.000000 and TNOM = 27.000000

Using SPARSE 1.3 as Direct Linear Solver
 Reference value :  0.00000e+00
No. of Data Rows : 1
ASCII raw file "LDO_op.raw"
Doing analysis at TEMP = 27.000000 and TNOM = 27.000000

Using SPARSE 1.3 as Direct Linear Solver

Initial Transient Solution
--------------------------

Node                                   Voltage
----                                   -------
x1.net1                                 1.0077
x1.net2                                1.02325
x1.x1.vo_ota1                         0.679099
net1                                       1.8
x1.x1.vtail                          0.0353661
x1.x1.vgp                             0.821029
vref                                       0.6
x1.net3                               0.601963
x1.x1.xr1.rb     

  .model x1.x1.xr1:rhead_model r sw_et=0 isnoisy=0 rsh=    3.4583120000000 ...
unrecognized parameter (sw_et) - ignored
unrecognized parameter (isnoisy) - ignored
unrecognized parameter (p2) - ignored
unrecognized parameter (q2) - ignored

  .model x1.x1.xr1:rbody_model r sw_et=0 isnoisy=0 rsh=    3.1738850000000 ...
unrecognized parameter (sw_et) - ignored
unrecognized parameter (isnoisy) - ignored
unrecognized parameter (p2) - ignored
unrecognized parameter (q2) - ignored
unrecognized parameter (p3) - ignored
unrecognized parameter (q3) - ignored

Error: -18293.3 out of range for sqrt
    in line b.x1.xr1.brpwres_1

Note: Starting dynamic gmin stepping
Error: -96927.7 out of range for sqrt
    in line b.x1.xr2.brpwres_1

Note: Dynamic gmin stepping completed
Error: -18293.3 out of range for sqrt
    in line b.x1.xr1.brpwres_1

Note: Starting dynamic gmin stepping
Error: -96927.7 out of range for sqrt
    in line b.x1.xr2.brpwres_1

Note: Dynamic gmin stepping completed
Error: -18293.

In [2]:
# ==============================================================================
# ENCABEZADO: Respuesta Transitoria (.TRAN) a Escalón de Carga
# DESCRIPCIÓN: Lee LDO_trans.raw y grafica Vout(t) ante variaciones de Iload(t).
# ==============================================================================

import pandas as pd

raw_trans_path = os.path.join(sim_dir, "LDO_trans.raw")
if not os.path.exists(raw_trans_path):
    raw_trans_path = os.path.join(xschem_dir, "LDO_trans.raw")

trans_data = read_ngspice_raw_ascii(raw_trans_path)

time_us = np.real(trans_data['time']) * 1e6
vout = np.real(trans_data['v(vo)'])

vout_dc = float(np.mean(vout[-100:]))
vout_min = float(np.min(vout))
vout_max = float(np.max(vout))
undershoot_mV = (vout_dc - vout_min) * 1e3
overshoot_mV = (vout_max - vout_dc) * 1e3

print("=== MÉTRICAS DE RESPUESTA TRANSITORIA DEL LDO ===")
print(f"Tensión Nominal de Salida (Vout): {vout_dc:.4f} V")
print(f"Caída Máxima (Undershoot):         {undershoot_mV:.2f} mV")
print(f"Sobrepico Máximo (Overshoot):       {overshoot_mV:.2f} mV")
print(f"Rango Dinámico de Tensión:         {vout_min:.4f} V a {vout_max:.4f} V")

fig_trans = go.Figure()
fig_trans.add_trace(go.Scatter(x=time_us, y=vout, mode='lines', name='Vout(t)', line=dict(color='#2ca02c', width=2.0)))
fig_trans.update_layout(
    title_text="<b>Respuesta Transitoria de Tensión de Salida Vout(t) ante Escalón de Carga</b>",
    xaxis_title="Tiempo [us]",
    yaxis_title="Tensión de Salida [V]",
    template="plotly_white",
    height=450
)
fig_trans.show()



=== MÉTRICAS DE RESPUESTA TRANSITORIA DEL LDO ===
Tensión Nominal de Salida (Vout): 1.1936 V
Caída Máxima (Undershoot):         6.96 mV
Sobrepico Máximo (Overshoot):       6.01 mV
Rango Dinámico de Tensión:         1.1867 V a 1.1996 V


In [3]:
# ==============================================================================
# ENCABEZADO: Punto de Operación (.OP) e Inspección del Transistor de Paso M1
# DESCRIPCIÓN: Evalúa la tensión Vout, Vref y estado del transistor de paso PMOS LVT.
# ==============================================================================

raw_op_path = os.path.join(sim_dir, "LDO_op.raw")
if not os.path.exists(raw_op_path):
    raw_op_path = os.path.join(xschem_dir, "LDO_op.raw")

op_data = read_ngspice_raw_ascii(raw_op_path)

vout_op = float(np.real(op_data['v(vo)'][0]))
vref_op = float(np.real(op_data['v(vref)'][0]))

print("=== PUNTO DE OPERACIÓN DC DEL LDO ===")
print(f"• Tensión de Referencia (Vref): {vref_op:.3f} V")
print(f"• Tensión Regula de Salida (Vout): {vout_op:.4f} V")



=== PUNTO DE OPERACIÓN DC DEL LDO ===
• Tensión de Referencia (Vref): 0.600 V
• Tensión Regula de Salida (Vout): 1.1936 V


In [4]:
# ==============================================================================
# ENCABEZADO: Cálculo del ÁREA TOTAL de Todo el Regulador LDO (Sky130)
# DESCRIPCIÓN: Escanea /headless/.xschem/simulations/LDO.spice para sumar el área
#              de TODOS los componentes: Transistor de Paso M1, Resistencias de
#              Realimentación, Capacitores de Compensación y el Amplificador OTA.
# ==============================================================================

import re
import pandas as pd
import numpy as np
import plotly.graph_objects as go

spice_file = os.path.join(sim_dir, "LDO.spice")
if not os.path.exists(spice_file):
    spice_file = os.path.join(xschem_dir, "LDO.spice")

with open(spice_file, 'r') as f:
    spice_content = f.read()

# Parsear subcircuitos LDO y OTA en el netlist
subckts = re.findall(r'\.subckt\s+(\w+).*?\n(.*?)\n\.ends', spice_content, re.DOTALL)

comp_list = []

for sub_name, sub_body in subckts:
    instances = re.findall(r'^\s*(X\w+|C\w+|R\w+)\s+(.*?)$', sub_body, re.MULTILINE)
    for name, params in instances:
        if name.startswith('p') or 'ipin' in params or 'opin' in params:
            continue
        clean_name = name[1:] if name.startswith('X') else name
        if clean_name.lower() in ('1', 'x1', 'ota'):
            continue
            
        w_m = re.search(r'\bW=([\d\.]+)', params, re.IGNORECASE)
        l_m = re.search(r'\bL=([\d\.]+)', params, re.IGNORECASE)
        nf_m = re.search(r'\bnf=([\d\.]+)', params, re.IGNORECASE)
        
        w = float(w_m.group(1)) if w_m else 1.0
        l = float(l_m.group(1)) if l_m else 0.5
        nf = int(nf_m.group(1)) if nf_m else 1
        
        m_matches = re.findall(r'\b(?:mult|m|MF)=([\d\.]+)', params, re.IGNORECASE)
        mult = float(m_matches[-1]) if m_matches else 1.0
        
        if clean_name.startswith('M1') or 'pfet_01v8_lvt' in params:
            tipo = 'Transistor Pass (PMOS LVT)'
        elif 'res' in params or clean_name.startswith('R'):
            tipo = 'Resistencia Poly'
        elif 'cap' in params or clean_name.startswith('C'):
            tipo = 'Capacitor MIM'
        else:
            tipo = 'Transistor MOS (Amplificador)'
            
        area_wl = w * l * mult
        comp_list.append({
            'Bloque': sub_name,
            'Componente': f'{sub_name}.{clean_name}',
            'Tipo': tipo,
            'W [um]': w,
            'L [um]': l,
            'nf': nf,
            'mult / MF': mult,
            'Área W×L [um²]': round(area_wl, 2)
        })

df_area_ldo = pd.DataFrame(comp_list)

area_pass = df_area_ldo[df_area_ldo['Tipo'] == 'Transistor Pass (PMOS LVT)']['Área W×L [um²]'].sum()
area_ota_mos = df_area_ldo[df_area_ldo['Tipo'] == 'Transistor MOS (Amplificador)']['Área W×L [um²]'].sum()
area_res_tot = df_area_ldo[df_area_ldo['Tipo'] == 'Resistencia Poly']['Área W×L [um²]'].sum()
area_cap_tot = df_area_ldo[df_area_ldo['Tipo'] == 'Capacitor MIM']['Área W×L [um²]'].sum()

area_total_bruta = area_pass + area_ota_mos + area_res_tot + area_cap_tot
factor_overhead = 1.25  # Overhead 25% por interconexiones, contactos y guardrings
area_total_layout = area_total_bruta * factor_overhead

print("=== TABLA DETALLADA DE COMPONENTES DEL REGULADOR LDO COMPLETO ===")
display(df_area_ldo)

print("\n=== RESUMEN GLOBAL DEL ÁREA TOTAL DE TODO EL LDO ===")
print(f"• Transistor de Paso M1 (PMOS LVT): {area_pass:8.2f} um²  ({(area_pass/area_total_bruta*100) if area_total_bruta else 0:5.1f}%)")
print(f"• Transistores MOS Amplificador:     {area_ota_mos:8.2f} um²  ({(area_ota_mos/area_total_bruta*100) if area_total_bruta else 0:5.1f}%)")
print(f"• Resistencias Poly (Realimentación):{area_res_tot:8.2f} um²  ({(area_res_tot/area_total_bruta*100) if area_total_bruta else 0:5.1f}%)")
print(f"• Capacitores MIM (Compensación):   {area_cap_tot:8.2f} um²  ({(area_cap_tot/area_total_bruta*100) if area_total_bruta else 0:5.1f}%)")
print(f"------------------------------------------------------------------")
print(f"• ÁREA TOTAL BRUTA DEL LDO (W×L):   {area_total_bruta:8.2f} um²")
print(f"• ÁREA TOTAL ESTIMADA LAYOUT (1.25x):{area_total_layout:8.2f} um²  ({area_total_layout/1e6:.6f} mm²)")
print(f"• Dimensión Cuadrada Est. Layout:   {np.sqrt(area_total_layout):.1f} um x {np.sqrt(area_total_layout):.1f} um")

pie_labels = ['Capacitores MIM', 'Transistor de Paso M1', 'Transistores OTA', 'Resistencias Poly']
pie_values = [area_cap_tot, area_pass, area_ota_mos, area_res_tot]

fig_pie = go.Figure(data=[go.Pie(
    labels=pie_labels,
    values=pie_values,
    hole=.3,
    marker_colors=['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e'],
    textinfo='label+percent+value',
    texttemplate='%{label}<br>%{value:.1f} um² (%{percent})'
)])

fig_pie.update_layout(
    title_text="<b>Desglose Porcentual del Área TOTAL del Regulador LDO Completo</b>",
    template="plotly_white",
    height=480
)
fig_pie.show()



=== TABLA DETALLADA DE COMPONENTES DEL REGULADOR LDO COMPLETO ===


,Bloque,Componente,Tipo,W [um],L [um],nf,mult / MF,Área W×L [um²]
0,LDO,LDO.M1,Transistor Pass (PMOS LVT),1000.00,0.35,50,1.0,350.00
1,LDO,LDO.R1,Resistencia Poly,0.35,10.00,1,1.0,3.50
2,LDO,LDO.R2,Resistencia Poly,0.35,10.00,1,1.0,3.50
3,LDO,LDO.C1,Capacitor MIM,56.00,55.00,1,1.0,3080.00
4,LDO,LDO.Mr,Transistor MOS (Amplificador),0.50,0.50,1,1.0,0.25
5,LDO,LDO.Mo,Transistor MOS (Amplificador),20.00,0.50,1,1.0,10.00
6,LDO,LDO.Mrc1,Transistor MOS (Amplificador),2.00,0.50,1,1.0,1.00
7,LDO,LDO.Mc2,Transistor MOS (Amplificador),1.00,0.50,1,1.0,0.50
8,LDO,LDO.Mn,Transistor MOS (Amplificador),40.00,0.50,1,1.0,20.00
9,LDO,LDO.Mp,Transistor MOS (Amplificador),40.00,0.50,1,1.0,20.00



=== RESUMEN GLOBAL DEL ÁREA TOTAL DE TODO EL LDO ===
• Transistor de Paso M1 (PMOS LVT):   350.00 um²  (  8.6%)
• Transistores MOS Amplificador:        91.75 um²  (  2.2%)
• Resistencias Poly (Realimentación):   62.65 um²  (  1.5%)
• Capacitores MIM (Compensación):    3586.25 um²  ( 87.7%)
------------------------------------------------------------------
• ÁREA TOTAL BRUTA DEL LDO (W×L):    4090.65 um²
• ÁREA TOTAL ESTIMADA LAYOUT (1.25x): 5113.31 um²  (0.005113 mm²)
• Dimensión Cuadrada Est. Layout:   71.5 um x 71.5 um
